# Exploratory Data Analysis

Student Performance Prediction project.


In [ ]:
# Session 14 - Study Time and Performance Analysis

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/student-mat.csv", sep=";")

analysis_df = df[["studytime", "G3"]].dropna().copy()

summary = (
    analysis_df
    .groupby("studytime")
    .agg(
        number_of_students=("G3", "count"),
        mean_G3=("G3", "mean"),
        median_G3=("G3", "median"),
        minimum_G3=("G3", "min"),
        maximum_G3=("G3", "max"),
    )
    .round(2)
)

display(summary)

print(
    "Monotonic increasing:",
    summary["mean_G3"].is_monotonic_increasing,
)

plt.figure(figsize=(8, 5.5))

sns.boxplot(
    data=analysis_df,
    x="studytime",
    y="G3",
)

plt.title("Final Grade by Study Time")
plt.xlabel("Study Time Group")
plt.ylabel("Final Grade (G3)")
plt.tight_layout()

plt.savefig(
    "../figures/studytime_vs_grade.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


## Session 14 Finding: Study Time and Final Grade

![Final Grade by Study Time](../figures/studytime_vs_grade.png)

The figure studytime_vs_grade.png compares final grade G3 across the four studytime groups.

### Mean Final Grade by Study-Time Group

- Studytime 1: mean G3 = 10.05; students = 105
- Studytime 2: mean G3 = 10.17; students = 198
- Studytime 3: mean G3 = 11.40; students = 65
- Studytime 4: mean G3 = 11.26; students = 27

The lowest mean final grade occurs at studytime level **1**.

The highest mean final grade occurs at studytime level **3**.

The trend is not perfectly monotonic because mean G3 decreases from studytime 3 to 4 (-0.14).

The boxplot and mean-grade results indicate an association between study time and final grade. However, this analysis does not establish that additional study time causes higher grades.

The dataset is observational. Prior grades, absences, previous failures, motivation, family support, school support, and other factors may affect both study behavior and academic performance.

A suitable follow-up is a regression or interpretable machine-learning analysis that examines whether studytime remains informative after accounting for other predictors.


In [ ]:
# SESSION16_AUTOMATION: support-group summaries
# Session 16: Family and school-support analysis

support_features = ["schoolsup", "famsup", "higher"]
required_columns = support_features + ["G3"]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required Session 16 columns: {missing_columns}"
    )

session16_df = df[required_columns].copy()

for feature in support_features:
    session16_df[feature] = (
        session16_df[feature]
        .astype(str)
        .str.strip()
        .str.lower()
    )

session16_df["G3"] = pd.to_numeric(
    session16_df["G3"],
    errors="coerce",
)

session16_df = session16_df.dropna(
    subset=required_columns,
)

support_summary_tables = []

for feature in support_features:
    summary = (
        session16_df
        .groupby(feature)["G3"]
        .agg(
            student_count="count",
            mean_G3="mean",
            median_G3="median",
            standard_deviation="std",
        )
        .round(2)
    )

    print("=" * 60)
    print(f"Final-grade summary by {feature}")
    display(summary)

    summary_table = (
        summary
        .reset_index()
        .rename(columns={feature: "response"})
    )
    summary_table.insert(0, "feature", feature)
    support_summary_tables.append(summary_table)

support_summary = pd.concat(
    support_summary_tables,
    ignore_index=True,
)

display(support_summary)


In [ ]:
# SESSION16_AUTOMATION: yes-versus-no differences

support_differences = []

for feature in ["schoolsup", "famsup", "higher"]:
    means = session16_df.groupby(feature)["G3"].mean()
    no_mean = means.get("no")
    yes_mean = means.get("yes")

    support_differences.append(
        {
            "feature": feature,
            "mean_G3_no": no_mean,
            "mean_G3_yes": yes_mean,
            "difference_yes_minus_no": yes_mean - no_mean,
        }
    )

support_difference_table = pd.DataFrame(
    support_differences
).round(2)

display(support_difference_table)


<!-- SESSION16_AUTOMATION -->
# Session 16: Family and School-Support Analysis

## Purpose

This exploratory analysis examines how school support, family support, and students' aspirations for higher education are associated with final academic performance. Final performance is measured using **G3**, the final course grade.

## Variables

- **schoolsup:** Whether the student receives additional educational support from the school.
- **famsup:** Whether the student receives educational support from the family.
- **higher:** Whether the student intends to pursue higher education.
- **G3:** Final course grade.

## Analysis Method

Students were grouped by the yes and no categories for schoolsup, famsup, and higher. The mean final grade, median final grade, standard deviation, and number of students were calculated for each group. This is a descriptive analysis of associations rather than causal effects.

## Numerical Results

The automated analysis used `data/student-mat.csv` and retained **395** complete analysis rows.

| Feature | Response | Student Count | Mean G3 | Median G3 | Standard Deviation |
|---|---:|---:|---:|---:|---:|
| schoolsup | no | 344 | 10.56 | 11.00 | 4.77 |
| schoolsup | yes | 51 | 9.43 | 10.00 | 2.87 |
| famsup | no | 153 | 10.64 | 11.00 | 4.64 |
| famsup | yes | 242 | 10.27 | 11.00 | 4.55 |
| higher | no | 20 | 6.80 | 8.00 | 4.83 |
| higher | yes | 375 | 10.61 | 11.00 | 4.49 |

| Feature | Mean G3: No | Mean G3: Yes | Difference: Yes - No |
|---|---:|---:|---:|
| schoolsup | 10.56 | 9.43 | -1.13 |
| famsup | 10.64 | 10.27 | -0.37 |
| higher | 6.80 | 10.61 | 3.81 |

## Summary

Students receiving school support had a mean G3 of **9.43**, compared with **10.56** for students not receiving school support.

Students receiving family support had a mean G3 of **10.27**, compared with **10.64** for students not receiving family support.

Students intending to pursue higher education had a mean G3 of **10.61**, compared with **6.80** for students not intending to pursue higher education.

## Interpretation

The school-support comparison showed a **lower** average final grade for students receiving school support.

The family-support comparison showed a **negative** descriptive relationship with final grades.

The higher-education comparison showed a **positive** descriptive relationship with final grades.

The strongest observed relationship was for **higher-education aspiration**, with an absolute mean-grade difference of **3.81** points.

The most counter-intuitive result was that **students receiving school support had a lower average final grade than students not receiving school support**. A plausible explanation is that **students with existing academic difficulties may have been more likely to receive school support**.

## Confounding Considerations

Prior academic performance may affect both whether a student receives support and the student's final grade. Other possible confounding variables include previous failures, attendance, study time, motivation, learning difficulties, family background, and socioeconomic conditions.

These findings represent **associations only** and do not establish causal effects. A lower mean grade among supported students would not prove that school support reduced performance. Support may have been assigned to students who were already experiencing academic difficulty.

## Recommendations

- Retain **schoolsup** because it may identify students receiving academic intervention or students with greater prior academic need.
- Retain **famsup** because it provides potentially useful family-context information.
- Retain **higher** because educational aspiration may reflect motivation, expectations, and future academic goals.
- Evaluate all three variables later in multivariable models rather than relying only on unadjusted group means.

## Reflection

If school support is associated with lower grades, a likely confounding explanation is selection into support. Students who were already struggling academically may have been more likely to receive assistance. Their lower final grades may reflect prior academic risk rather than a harmful effect of the support itself.

## Conclusion

School support, family support, and higher-education aspiration showed different descriptive relationships with final grades. The largest unadjusted difference was associated with **higher-education aspiration**, while the counter-intuitive finding was that **students receiving school support had a lower average final grade than students not receiving school support**.

All three features should be retained for later multivariable analysis, with careful attention to confounding and the distinction between association and causation.
